In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("CREDIT RISK ASSESSMENT - DATA CLEANING")
print("=" * 60)

# ============================================
# LOAD THE RAW DATA
# ============================================

print("\n Loading raw data...")
df = pd.read_csv('C:/Users/AlexB/Downloads/cs-training.csv', index_col=0)
print(f" Raw data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

# Create a copy for cleaning
df_clean = df.copy()


CREDIT RISK ASSESSMENT - DATA CLEANING

 Loading raw data...
 Raw data loaded: 150,000 rows, 11 columns


In [14]:
# 1. REMOVE DUPLICATES
# ============================================

print("\n" + "-" * 40)
print("1. REMOVING DUPLICATES")
print("-" * 40)

initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
duplicates_removed = initial_rows - len(df_clean)
print(f" Removed {duplicates_removed} duplicate rows")
print(f"   Rows before: {initial_rows:,}")
print(f"   Rows after: {len(df_clean):,}")


----------------------------------------
1. REMOVING DUPLICATES
----------------------------------------
 Removed 609 duplicate rows
   Rows before: 150,000
   Rows after: 149,391


In [15]:
# ============================================
# 2. FIX AGE ISSUES
# ============================================

print("\n" + "-" * 40)
print("2. FIXING AGE ISSUES")
print("-" * 40)

# Fix age = 0 (replace with median age)
median_age = df_clean[df_clean['age'] > 0]['age'].median()
zero_age_count = (df_clean['age'] == 0).sum()
df_clean.loc[df_clean['age'] == 0, 'age'] = median_age
print(f" Fixed {zero_age_count} row(s) with age = 0")
print(f"   Replaced with median age: {median_age}")

# Cap age > 100 at 100
old_age_count = (df_clean['age'] > 100).sum()
df_clean.loc[df_clean['age'] > 100, 'age'] = 100
print(f" Capped {old_age_count} row(s) with age > 100")


----------------------------------------
2. FIXING AGE ISSUES
----------------------------------------
 Fixed 1 row(s) with age = 0
   Replaced with median age: 52.0
 Capped 13 row(s) with age > 100


In [16]:
# Check age range after fixes
print(f"   Age range after fixes: {df_clean['age'].min()} to {df_clean['age'].max()}")

   Age range after fixes: 21 to 100


In [17]:
# 3. FIX CREDIT UTILIZATION
# ============================================

print("\n" + "-" * 40)
print("3. FIXING CREDIT UTILIZATION")
print("-" * 40)

# Cap utilization at 1.0 (100%)
high_util_count = (df_clean['RevolvingUtilizationOfUnsecuredLines'] > 1).sum()
df_clean.loc[df_clean['RevolvingUtilizationOfUnsecuredLines'] > 1, 'RevolvingUtilizationOfUnsecuredLines'] = 1.0
print(f" Capped {high_util_count:,} row(s) with utilization > 100%")

# Check utilization range after fixes
print(f"   Utilization range after fixes: {df_clean['RevolvingUtilizationOfUnsecuredLines'].min():.2f} to {df_clean['RevolvingUtilizationOfUnsecuredLines'].max():.2f}")


----------------------------------------
3. FIXING CREDIT UTILIZATION
----------------------------------------
 Capped 3,321 row(s) with utilization > 100%
   Utilization range after fixes: 0.00 to 1.00


In [18]:
# ============================================
# 4. FIX PAST DUE PLACEHOLDERS (98 values)
# ============================================

print("\n" + "-" * 40)
print("4. FIXING PAST DUE PLACEHOLDERS")
print("-" * 40)

# For 30-59 days past due
past_due_30_count = (df_clean['NumberOfTime30-59DaysPastDueNotWorse'] == 98).sum()
df_clean.loc[df_clean['NumberOfTime30-59DaysPastDueNotWorse'] == 98, 'NumberOfTime30-59DaysPastDueNotWorse'] = 0
print(f" Fixed {past_due_30_count:,} row(s) with 30-59 days past due = 98")

# For 60-89 days past due
past_due_60_count = (df_clean['NumberOfTime60-89DaysPastDueNotWorse'] == 98).sum()
df_clean.loc[df_clean['NumberOfTime60-89DaysPastDueNotWorse'] == 98, 'NumberOfTime60-89DaysPastDueNotWorse'] = 0
print(f" Fixed {past_due_60_count:,} row(s) with 60-89 days past due = 98")
# For 90+ days past due
past_due_90_count = (df_clean['NumberOfTimes90DaysLate'] == 98).sum()
df_clean.loc[df_clean['NumberOfTimes90DaysLate'] == 98, 'NumberOfTimes90DaysLate'] = 0
print(f" Fixed {past_due_90_count:,} row(s) with 90+ days past due = 98")


----------------------------------------
4. FIXING PAST DUE PLACEHOLDERS
----------------------------------------
 Fixed 220 row(s) with 30-59 days past due = 98
 Fixed 220 row(s) with 60-89 days past due = 98
 Fixed 220 row(s) with 90+ days past due = 98


In [19]:
# 5. FIX DEBT RATIO (cap extreme outliers)
# ============================================

print("\n" + "-" * 40)
print("5. FIXING DEBT RATIO")
print("-" * 40)

debt_99th = df_clean['DebtRatio'].quantile(0.99)
debt_capped_count = (df_clean['DebtRatio'] > debt_99th).sum()
df_clean.loc[df_clean['DebtRatio'] > debt_99th, 'DebtRatio'] = debt_99th
print(f" Capped {debt_capped_count:,} row(s) with extreme debt ratio")
print(f"   99th percentile value: {debt_99th:.2f}")
print(f"   Debt ratio range after capping: {df_clean['DebtRatio'].min():.2f} to {df_clean['DebtRatio'].max():.2f}")


----------------------------------------
5. FIXING DEBT RATIO
----------------------------------------
 Capped 1,494 row(s) with extreme debt ratio
   99th percentile value: 4985.10
   Debt ratio range after capping: 0.00 to 4985.10


In [20]:
# ============================================
# 6. FIX MONTHLY INCOME
# ============================================

print("\n" + "-" * 40)
print("6. FIXING MONTHLY INCOME")
print("-" * 40)

# Calculate median income (excluding zeros and negatives)
median_income = df_clean[df_clean['MonthlyIncome'] > 0]['MonthlyIncome'].median()
print(f" Median Monthly Income: ${median_income:,.2f}")

# Fix zero/negative income
bad_income_count = (df_clean['MonthlyIncome'] <= 0).sum()
df_clean.loc[df_clean['MonthlyIncome'] <= 0, 'MonthlyIncome'] = median_income
print(f" Fixed {bad_income_count:,} row(s) with zero/negative income")

# Fill missing income
missing_income_before = df_clean['MonthlyIncome'].isnull().sum()
df_clean['MonthlyIncome'] = df_clean['MonthlyIncome'].fillna(median_income)
print(f" Filled {missing_income_before:,} missing MonthlyIncome values")

# Check income after fixes
print(f"   Income range after fixes: ${df_clean['MonthlyIncome'].min():,.2f} to ${df_clean['MonthlyIncome'].max():,.2f}")


----------------------------------------
6. FIXING MONTHLY INCOME
----------------------------------------
 Median Monthly Income: $5,443.00
 Fixed 1,616 row(s) with zero/negative income
 Filled 29,221 missing MonthlyIncome values
   Income range after fixes: $1.00 to $3,008,750.00


In [21]:
# ============================================
# 7. FIX NUMBER OF DEPENDENTS
# ============================================

print("\n" + "-" * 40)
print("7. FIXING NUMBER OF DEPENDENTS")
print("-" * 40)

median_dependents = df_clean['NumberOfDependents'].median()
print(f" Median Number of Dependents: {median_dependents}")

missing_dependents_before = df_clean['NumberOfDependents'].isnull().sum()
df_clean['NumberOfDependents'] = df_clean['NumberOfDependents'].fillna(median_dependents)
print(f" Filled {missing_dependents_before:,} missing NumberOfDependents values")

# Check dependents range
print(f"   Dependents range: {df_clean['NumberOfDependents'].min()} to {df_clean['NumberOfDependents'].max()}")



----------------------------------------
7. FIXING NUMBER OF DEPENDENTS
----------------------------------------
 Median Number of Dependents: 0.0
 Filled 3,828 missing NumberOfDependents values
   Dependents range: 0.0 to 20.0


In [22]:
# ============================================
# 8. CAP OTHER OUTLIERS
# ============================================

print("\n" + "-" * 40)
print("8. CAPPING OTHER OUTLIERS")
print("-" * 40)

# Number of open credit lines
credit_lines_99th = df_clean['NumberOfOpenCreditLinesAndLoans'].quantile(0.99)
credit_lines_capped = (df_clean['NumberOfOpenCreditLinesAndLoans'] > credit_lines_99th).sum()
df_clean.loc[df_clean['NumberOfOpenCreditLinesAndLoans'] > credit_lines_99th, 'NumberOfOpenCreditLinesAndLoans'] = credit_lines_99th
print(f" Capped {credit_lines_capped:,} row(s) with extreme number of credit lines")
print(f"   99th percentile: {credit_lines_99th:.0f}")

# Real estate loans
real_estate_99th = df_clean['NumberRealEstateLoansOrLines'].quantile(0.99)
real_estate_capped = (df_clean['NumberRealEstateLoansOrLines'] > real_estate_99th).sum()
df_clean.loc[df_clean['NumberRealEstateLoansOrLines'] > real_estate_99th, 'NumberRealEstateLoansOrLines'] = real_estate_99th
print(f" Capped {real_estate_capped:,} row(s) with extreme real estate loans")
print(f"   99th percentile: {real_estate_99th:.0f}")


----------------------------------------
8. CAPPING OTHER OUTLIERS
----------------------------------------
 Capped 1,476 row(s) with extreme number of credit lines
   99th percentile: 24
 Capped 1,482 row(s) with extreme real estate loans
   99th percentile: 4


In [23]:
# ============================================
# VERIFY CLEANING RESULTS
# ============================================

print("\n" + "=" * 60)
print("VERIFY CLEANING RESULTS")
print("=" * 60)

print(f"\n Dataset shape after cleaning: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns")
print(f"\n Missing values after cleaning: {df_clean.isnull().sum().sum():,}")

print("\n Invalid values check:")
print(f"   - Age = 0: {(df_clean['age'] == 0).sum()}")
print(f"   - Age > 100: {(df_clean['age'] > 100).sum()}")
print(f"   - Utilization > 1: {(df_clean['RevolvingUtilizationOfUnsecuredLines'] > 1).sum()}")
print(f"   - Past due = 98: {(df_clean['NumberOfTime30-59DaysPastDueNotWorse'] == 98).sum()}")
print(f"   - MonthlyIncome <= 0: {(df_clean['MonthlyIncome'] <= 0).sum()}")
print(f"   - MonthlyIncome missing: {df_clean['MonthlyIncome'].isnull().sum()}")
print(f"   - Dependents missing: {df_clean['NumberOfDependents'].isnull().sum()}")


VERIFY CLEANING RESULTS

 Dataset shape after cleaning: 149,391 rows, 11 columns

 Missing values after cleaning: 0

 Invalid values check:
   - Age = 0: 0
   - Age > 100: 0
   - Utilization > 1: 0
   - Past due = 98: 0
   - MonthlyIncome <= 0: 0
   - MonthlyIncome missing: 0
   - Dependents missing: 0


In [24]:
import os

print("\n" + "=" * 60)
print("SAVING CLEANED DATA")
print("=" * 60)

# Create the directory if it doesn't exist (absolute path)
save_dir = 'C:/Users/AlexB/Desktop/credit_risk_assessment/data/processed'
os.makedirs(save_dir, exist_ok=True)

# Save to processed folder (absolute path)
output_path = 'C:/Users/AlexB/Desktop/credit_risk_assessment/data/processed/credit_data_cleaned.csv'
df_clean.to_csv(output_path, index=False)
print(f" Cleaned data saved to: {output_path}")

# Also save a copy to Desktop for easy access
desktop_path = 'C:/Users/AlexB/Desktop/credit_data_cleaned.csv'
df_clean.to_csv(desktop_path, index=False)
print(f" Cleaned data also saved to: {desktop_path}")

# ============================================
# SUMMARY REPORT
# ============================================

print("\n" + "=" * 60)
print("CLEANING SUMMARY REPORT")
print("=" * 60)

print(f"""
 Cleaning Summary:
   - Original rows: 150,000
   - Final rows: {len(df_clean):,}
   - Duplicates removed: 609
   
 Fixes Applied:
   - Age = 0 fixed: 1
   - Age > 100 capped: 13
   - Utilization > 100% capped: 3,321
   - Past due 98 fixed: 792
   - Debt ratio outliers capped: {debt_capped_count:,}
   - Income issues fixed: {bad_income_count + missing_income_before:,}
   - Dependents missing filled: {missing_dependents_before:,}
   - Credit lines outliers capped: {credit_lines_capped:,}
   - Real estate loans capped: {real_estate_capped:,}

 Data cleaning complete! Ready for EDA and Modeling.
""")

print("=" * 60)
print("DATA CLEANING COMPLETE!")
print("=" * 60)


SAVING CLEANED DATA
 Cleaned data saved to: C:/Users/AlexB/Desktop/credit_risk_assessment/data/processed/credit_data_cleaned.csv
 Cleaned data also saved to: C:/Users/AlexB/Desktop/credit_data_cleaned.csv

CLEANING SUMMARY REPORT

 Cleaning Summary:
   - Original rows: 150,000
   - Final rows: 149,391
   - Duplicates removed: 609

 Fixes Applied:
   - Age = 0 fixed: 1
   - Age > 100 capped: 13
   - Utilization > 100% capped: 3,321
   - Past due 98 fixed: 792
   - Debt ratio outliers capped: 1,494
   - Income issues fixed: 30,837
   - Dependents missing filled: 3,828
   - Credit lines outliers capped: 1,476
   - Real estate loans capped: 1,482

 Data cleaning complete! Ready for EDA and Modeling.

DATA CLEANING COMPLETE!
